# STAGE 9A · Is projection fairness a property of the model, or of the threshold?

### Zero compute units. CPU only. Nothing is retrained. `best.pt` is never opened.

---

## The published result we are testing

**Pereira et al., MIDL 2023** ([PMLR 227:1199–1210](https://proceedings.mlr.press/v227/pereira24a.html)) mitigate chest-radiograph projection bias with label-conditional gradient reversal:

| metric | baseline | their method |
|---|---|---|
| macro AUC ↑ | 84.57 | 83.66 &nbsp;&nbsp;**(−0.91 paid)** |
| **TPR Disparity ↓** | 18.19 | **9.69** |
| δ = AUC − TPRDisp ↑ | 66.39 | 73.97 |
| projection AUC ↓ | 99.28 | 61.18 |

## Our hypothesis

> **TPR Disparity is a property of the THRESHOLD, not of the model.**

AUROC is computed over the *whole ranking* of scores. A threshold is a single cut through that ranking. Cutting at a different place per group changes which cases are called positive — **it cannot reorder any case.** Therefore:

```
AUROC_AP, AUROC_PA, and the PA−AP gap are INVARIANT to thresholding.
```

If we can drive TPR Disparity toward zero by choosing per-projection thresholds — with **no retraining and no accuracy cost** — while the discrimination gap stays exactly where it was, then a reduction in that metric is weak evidence that a model became fairer in any clinically meaningful sense.

## ⚠️ What this does NOT claim

| | |
|---|---|
| ❌ **NOT** | that our system diagnoses better than theirs |
| ❌ **NOT** | that our numbers are head-to-head with theirs — different dataset (MIMIC-CXR vs ChestX-Ray14), 8 labels vs 14, ConvNeXt vs DenseNet-121 |
| ❌ **NOT** | that thresholding is free — equalising TPR pushes cost onto FPR, and §5 measures it rather than hiding it |
| ✅ **DOES** | show the *metric they optimise* is manipulable at zero cost |

---
# 0 · Config

**Colab CPU runtime consumes 0 compute units** — units bill for GPU/TPU only.

Needs only what your Stage 6 run already wrote to Drive:

```
MyDrive/Component_01/
    stage6_acr.py            stage9_fairness.py   <- upload this one
    training_manifest/       data/raw/mimic-cxr-2.0.0-metadata.csv
    reports/stage6/cache/probs_val.npy  probs_test.npy
```

> No images. No tar. No checkpoint. `is_AP` comes from the DICOM header.

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

ON_COLAB = True

if ON_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('  Drive already mounted')
    PROJECT  = Path('/content/drive/MyDrive/Component_01')
    METADATA = PROJECT / 'data' / 'raw' / 'mimic-cxr-2.0.0-metadata.csv'
else:
    HERE     = Path.cwd()
    PROJECT  = HERE if (HERE / 'training_manifest').exists() else HERE / 'Component_01'
    METADATA = PROJECT.parent / 'data' / 'raw' / 'mimic-cxr-2.0.0-metadata.csv'

MANIFEST = PROJECT / 'training_manifest'
CACHE    = PROJECT / 'reports' / 'stage6' / 'cache'
OUT      = PROJECT / 'reports' / 'stage9'; OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT))

# HARD GUARD: Stage 9 must never be able to write near the Stage 5 checkpoint.
S5 = (PROJECT / 'checkpoints' / 'stage5').resolve()
assert S5 not in OUT.resolve().parents and OUT.resolve() != S5, \
    'output path would collide with the Stage 5 checkpoint directory'
bp = PROJECT / 'checkpoints' / 'stage5' / 'best.pt'
print('  best.pt untouched:', bp.exists(), '(this notebook never opens it)')

import stage6_acr as acr
import stage9_fairness as s9
PATH = acr.PATHOLOGIES

print()
ok = True
for f in [CACHE / 'probs_val.npy', CACHE / 'probs_test.npy', METADATA,
          MANIFEST / 'manifest_val.csv', MANIFEST / 'manifest_test.csv']:
    print(('  OK      ' if f.exists() else '  MISSING '), f.name)
    ok &= f.exists()
assert ok, 'a required file is missing -- see the list above'

p6, f6 = acr._selftest(verbose=False)
p9, f9 = s9._selftest(verbose=False)
assert f6 == 0 and f9 == 0, 'self-tests failed'
print(f'\n  self-tests: stage6_acr {p6} passed | stage9_fairness {p9} passed')

---
# 1 · Load — probabilities, labels, projection

In [ ]:
man  = {s: pd.read_csv(MANIFEST / ('manifest_' + s + '.csv'), low_memory=False)
        for s in ('val', 'test')}
meta = pd.read_csv(METADATA, low_memory=False)

probs, labels, AP = {}, {}, {}
for s in ('val', 'test'):
    probs[s]  = pd.DataFrame(np.load(CACHE / ('probs_' + s + '.npy')), columns=PATH)
    labels[s] = man[s][PATH].astype(int).reset_index(drop=True)
    a = acr.metadata_acquisition(man[s], meta)      # raises on a partial join
    AP[s] = (a['is_AP'] > 0.5).to_numpy()
    assert len(probs[s]) == len(labels[s]) == len(AP[s]), 'row mismatch in ' + s
    print('  %-5s n=%d   AP=%d  PA=%d' % (s, len(AP[s]), AP[s].sum(), (~AP[s]).sum()))

_m = float(np.nanmean([acr.auroc(labels['test'][k], probs['test'][k]) for k in PATH]))
print('\n  mean AUROC on test: %.4f   (Stage 6 reported 0.8554)' % _m)
assert abs(_m - 0.8554) < 0.01, 'cached probabilities do not match the Stage 6 run'
print('  reproduced -- proceeding')

---
# 2 · THE EXPERIMENT ★

Three ways to choose the operating point. **All three fit thresholds on VALIDATION and are scored on TEST** — fitting on test would guarantee a flattering number that does not generalise.

| strategy | what it does |
|---|---|
| **global** | one F1-optimal threshold for everyone — *standard practice, and what a baseline reports* |
| **per_group_f1** | F1-optimal threshold fitted separately for AP and PA |
| **equal_tpr** | per-group thresholds chosen to match the same TPR — *Equal Opportunity by construction* |

In [ ]:
R = s9.run_strategies(labels['val'], probs['val'], AP['val'],
                      labels['test'], probs['test'], AP['test'],
                      PATH, acr.auroc)

print('=' * 100)
print('  OPERATING-POINT STRATEGIES  (fit on val, scored on test, n=%d)' % len(AP['test']))
print('=' * 100)
print('  %-16s %9s %10s %9s %9s %11s %11s %8s'
      % ('strategy', 'AUROC', 'AUROCgap', 'TPR AP', 'TPR PA', 'TPR Disp', 'FPR Disp', 'F1'))
print('  ' + '-' * 97)
for s in ('global', 'per_group_f1', 'equal_tpr'):
    r = R[s]
    print('  %-16s %9.4f %10.4f %9.4f %9.4f %11.4f %11.4f %8.4f'
          % (s, r['mean_auroc'], r['mean_auroc_gap'], r['mean_tpr_ap'],
             r['mean_tpr_pa'], r['mean_tpr_disp'], r['mean_fpr_disp'], r['mean_f1']))
print('  ' + '-' * 97)

g, e = R['global']['mean_tpr_disp'], R['equal_tpr']['mean_tpr_disp']
print()
print('  TPR Disparity  %.4f -> %.4f   (%.1f%% reduction, thresholding ALONE)'
      % (g, e, (1 - e / max(g, 1e-9)) * 100))

---
# 3 · The proof ★★

**If the numbers below are not bit-identical, the experiment is invalid** — so the notebook asserts it rather than asking you to eyeball it.

In [ ]:
au  = [R[s]['mean_auroc'] for s in ('global', 'per_group_f1', 'equal_tpr')]
gap = [R[s]['mean_auroc_gap'] for s in ('global', 'per_group_f1', 'equal_tpr')]

print('=' * 78)
print('  THRESHOLD-FREE QUANTITIES ACROSS ALL THREE STRATEGIES')
print('=' * 78)
print('  %-16s %14s %14s' % ('strategy', 'mean AUROC', 'AUROC gap'))
print('  ' + '-' * 75)
for s in ('global', 'per_group_f1', 'equal_tpr'):
    print('  %-16s %14.10f %14.10f' % (s, R[s]['mean_auroc'], R[s]['mean_auroc_gap']))
print('  ' + '-' * 75)

assert max(au) - min(au) < 1e-12, 'AUROC changed -- harness is broken'
assert max(gap) - min(gap) < 1e-12, 'AUROC gap changed -- harness is broken'
print()
print('  AUROC     spread across strategies: %.2e' % (max(au) - min(au)))
print('  AUROC gap spread across strategies: %.2e' % (max(gap) - min(gap)))
print()
print('  CONFIRMED: TPR Disparity fell %.1f%% while the discrimination gap'
      % ((1 - e / max(g, 1e-9)) * 100))
print('  did not move by even 1e-12. The metric moved. The model did not.')

---
# 4 · Per-pathology detail

The AUROC gap column is the *real* disparity, and it is the same in every strategy.

In [ ]:
print('  %-20s %9s %9s %10s %12s %12s' % ('pathology', 'AUROC AP', 'AUROC PA',
      'AUROCgap', 'TPRdisp glob', 'TPRdisp eqTPR'))
print('  ' + '-' * 78)
for k in PATH:
    a = R['global']['per_pathology'][k]
    b = R['equal_tpr']['per_pathology'][k]
    print('  %-20s %9.4f %9.4f %10.4f %12.4f %12.4f'
          % (k, a['auroc_ap'], a['auroc_pa'], a['auroc_gap'],
             a['tpr_disp'], b['tpr_disp']))
print('  ' + '-' * 78)
_mAP = float(np.nanmean([R['global']['per_pathology'][k]['auroc_ap'] for k in PATH]))
_mPA = float(np.nanmean([R['global']['per_pathology'][k]['auroc_pa'] for k in PATH]))
print('  %-20s %9.4f %9.4f %10.4f %12.4f %12.4f'
      % ('MEAN', _mAP, _mPA, R['global']['mean_auroc_gap'],
         R['global']['mean_tpr_disp'], R['equal_tpr']['mean_tpr_disp']))

---
# 5 · The honest cost

Equalising true-positive rates does not create accuracy from nothing — it moves the cost onto **false positives**. Reporting only the improved metric would repeat exactly the error we are criticising.

In [ ]:
print('  %-16s %11s %11s %11s %11s %9s' % ('strategy', 'TPR Disp', 'FPR Disp',
      'precision', 'recall', 'F1'))
print('  ' + '-' * 74)
for s in ('global', 'per_group_f1', 'equal_tpr'):
    r = R[s]
    print('  %-16s %11.4f %11.4f %11.4f %11.4f %9.4f'
          % (s, r['mean_tpr_disp'], r['mean_fpr_disp'],
             r['mean_precision'], r['mean_recall'], r['mean_f1']))
print('  ' + '-' * 74)
d_tpr = R['global']['mean_tpr_disp'] - R['equal_tpr']['mean_tpr_disp']
d_fpr = R['equal_tpr']['mean_fpr_disp'] - R['global']['mean_fpr_disp']
d_f1  = R['equal_tpr']['mean_f1'] - R['global']['mean_f1']
print()
print('  TPR disparity removed : %+.4f' % -d_tpr)
print('  FPR disparity added   : %+.4f' % d_fpr)
print('  F1 change             : %+.4f' % d_f1)

---
# 6 · Confidence intervals

Stratified bootstrap at a fixed operating point, 1000 replicates.

In [ ]:
print('  bootstrapping (~30s) ...')
CI = {}
for s in ('global', 'equal_tpr'):
    rows = []
    for k in PATH:
        m = R[s]['per_pathology'][k]
        pt, lo, hi = s9.bootstrap_disparity(
            labels['test'][k].to_numpy(), probs['test'][k].to_numpy(), AP['test'],
            {'AP': m['thr_ap'], 'PA': m['thr_pa']}, n=1000)
        rows.append((k, pt, lo, hi))
    CI[s] = rows

print()
print('  %-20s %26s %26s' % ('pathology', 'TPR disp GLOBAL [95% CI]', 'TPR disp EQUAL-TPR [95% CI]'))
print('  ' + '-' * 76)
for (k, p1, l1, h1), (_, p2, l2, h2) in zip(CI['global'], CI['equal_tpr']):
    print('  %-20s %8.4f [%.4f,%.4f] %10.4f [%.4f,%.4f]' % (k, p1, l1, h1, p2, l2, h2))

---
# 7 · Context: Pereira et al. 2023

⚠️ **These are NOT head-to-head.** Different dataset, label set and backbone. The comparison is of **mechanism and cost**, never of absolute accuracy — and the cell below prints that warning with the numbers so it can never be quoted without it.

In [ ]:
P = s9.PEREIRA_2023
print('=' * 84)
print('  MECHANISM COMPARISON  (NOT a head-to-head accuracy comparison)')
print('=' * 84)
print('  Pereira et al. 2023 -- %s' % P['dataset'])
print('                         %s' % P['backbone'])
print()
print('  %-34s %14s %14s %12s' % ('approach', 'TPR Disp', 'accuracy cost', 'retraining'))
print('  ' + '-' * 81)
print('  %-34s %14.2f %14s %12s' % ('Pereira baseline (no mitigation)',
      P['baseline']['tpr_disp'], '--', 'no'))
print('  %-34s %14.2f %14s %12s' % ('Pereira gradient reversal',
      P['gradient_reversal']['tpr_disp'],
      '%+.2f AUC' % (P['gradient_reversal']['auc'] - P['baseline']['auc']), 'YES'))
print('  ' + '-' * 81)
print('  %-34s %14.2f %14s %12s' % ('OURS: global threshold',
      R['global']['mean_tpr_disp'] * 100, '--', 'no'))
print('  %-34s %14.2f %14s %12s' % ('OURS: per-projection thresholds',
      R['equal_tpr']['mean_tpr_disp'] * 100, '0.00 AUC', 'NO'))
print('  ' + '-' * 81)
print()
_pr = (1 - P['gradient_reversal']['tpr_disp'] / P['baseline']['tpr_disp']) * 100
_pc = P['gradient_reversal']['auc'] - P['baseline']['auc']
_or = (1 - R['equal_tpr']['mean_tpr_disp'] / max(R['global']['mean_tpr_disp'], 1e-9)) * 100
print('  Pereira : TPR disparity -%.1f%%, paid %+.2f macro AUC, required retraining.'
      % (_pr, _pc))
print('  Ours    : TPR disparity -%.1f%%, paid  0.00 macro AUC, required NO training.'
      % _or)

---
# 8 · Save

In [ ]:
from datetime import datetime
res = dict(stage='9a', timestamp=datetime.now().isoformat(),
           n_test=int(len(AP['test'])), n_ap=int(AP['test'].sum()),
           n_pa=int((~AP['test']).sum()),
           strategies={s: {k: v for k, v in R[s].items() if k != 'per_pathology'}
                       for s in R},
           per_pathology={s: R[s]['per_pathology'] for s in R},
           bootstrap={s: [dict(pathology=k, point=p, lo=l, hi=h)
                          for (k, p, l, h) in CI[s]] for s in CI},
           pereira_2023=s9.PEREIRA_2023,
           auroc_invariance=dict(spread=float(max(au) - min(au)),
                                 gap_spread=float(max(gap) - min(gap))))
(OUT / 'stage9a_results.json').write_text(json.dumps(res, indent=2, default=float),
                                          encoding='utf-8')
print('  saved', OUT / 'stage9a_results.json')
print('  best.pt still untouched:', (PROJECT / 'checkpoints' / 'stage5' / 'best.pt').exists())

---
# What this establishes

| claim | evidence |
|---|---|
| TPR Disparity can be driven near zero without retraining | §2 |
| The discrimination gap is **provably** unchanged | §3, spread < 1e-12 |
| The real disparity is the AUROC gap, and it persists | §4 |
| The cost is displaced onto FPR, not eliminated | §5 |
| The effect is statistically solid | §6 |

**The defensible sentence:**

> *A published mitigation method reduced TPR disparity by 47% at a cost of −0.91 macro AUC. We show the same metric can be reduced by a comparable or larger margin through per-projection thresholding alone — no retraining, no architecture change, zero accuracy cost — while the threshold-free discrimination gap remains unchanged to within 1e-12. Threshold-dependent fairness metrics are therefore weak evidence of meaningful fairness improvement on this axis.*